In [1]:
import os
import pandas as pd
from lxml import etree

def list_files_in_directory(directory):
    # Lista para almacenar detalles de los archivos
    file_list = []
    
    # Recorrer el directorio y subdirectorios
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.xml'):
                file_path = os.path.join(root, file)

                try:
                    tree = etree.parse(file_path)
                    root_element = tree.getroot()
                
                    namespaces = {
                        'cfdi': 'http://www.sat.gob.mx/cfd/4',
                        'pago20': 'http://www.sat.gob.mx/Pagos20'
                    }

                    fecha_comprobante = root_element.get('Fecha', 'N/A')

                    receptord = root_element.find('.//cfdi:Receptor', namespaces)
                    rfcr = receptord.get('Rfc', 'N/A') if receptord is not None else 'N/A'
                    nombrer = receptord.get('Nombre', 'N/A') if receptord is not None else 'N/A'
                    regr = receptord.get('RegimenFiscalReceptor', 'N/A') if receptord is not None else 'N/A'

                    pagos = root_element.findall('.//pago20:Pago', namespaces)
                    
                    for pago in pagos:
                        fechaP = pago.get('FechaPago', 'N/A')
                        formaP = pago.get('FormaDePagoP', 'N/A')
                        monedaP = pago.get('MonedaP', 'N/A')
                        tipo_cP = pago.get('TipoCambioP', 'N/A')
                        montoP = pago.get('Monto', 'N/A')
                        nooperP = pago.get('NumOperacion', 'N/A')
                        bcoordP = pago.get('NomBancoOrdExt', 'N/A')
                        ctaordP = pago.get('CtaOrdenante', 'N/A')
                        ctabenP = pago.get('CtaBeneficiario', 'N/A')
                    
                        doctos_relacionados = pago.findall('.//pago20:DoctoRelacionado', namespaces)
                    
                        for docto in doctos_relacionados:
                            id_documento = docto.get('IdDocumento', 'N/A')
                            serie = docto.get('Serie', 'N/A')
                            folio = docto.get('Folio', 'N/A')
                            moneda_dr = docto.get('MonedaDR', 'N/A')
                            num_parcialidad = docto.get('NumParcialidad', 'N/A')
                            imp_saldo_ant = docto.get('ImpSaldoAnt', 'N/A')
                            imp_pagado = docto.get('ImpPagado', 'N/A')
                            imp_saldo_insoluto = docto.get('ImpSaldoInsoluto', 'N/A')

                            # Buscar traslados por documento relacionado
                            trasladosDR = docto.findall('.//pago20:TrasladoDR', namespaces)
                            
                            if trasladosDR:
                                for traslado in trasladosDR:
                                    file_list.append({
                                        "fecha_comprobante": fecha_comprobante,
                                        "RFC": rfcr,
                                        "E/R Nombre": nombrer,
                                        "Reg Fiscal": regr,
                                        "Archivo": file,
                                        "UUID": file,
                                        "Fecha Pago": fechaP,
                                        "Forma Pago": formaP,
                                        "Moneda Pago": monedaP,
                                        "Tipo_C Pago": tipo_cP,
                                        "Monto Pago": montoP,
                                        "# Operac Pago": nooperP,
                                        "Banco Pago": bcoordP,
                                        "Cta ordenante Pago": ctaordP,
                                        "Cta beneficiario Pago": ctabenP,
                                        "ID Doct": id_documento,
                                        "Serie": serie,
                                        "Folio": folio,
                                        "Moneda": moneda_dr,
                                        "Parcialidad": num_parcialidad,
                                        "ImpSaldoAnt": imp_saldo_ant,
                                        "ImpPagado": imp_pagado,
                                        "ImpSaldoInsoluto": imp_saldo_insoluto,
                                        "BaseDR": traslado.get('BaseDR', 'N/A'),
                                        "ImporteDR": traslado.get('ImporteDR', 'N/A'),
                                        "ImpuestoDR": traslado.get('ImpuestoDR', 'N/A')
                                    })
                            else:
                                # Si no tiene traslados, de todas formas agregamos el docto
                                file_list.append({
                                    "fecha_comprobante": fecha_comprobante,
                                    "RFC": rfcr,
                                    "E/R Nombre": nombrer,
                                    "Reg Fiscal": regr,
                                    "Archivo": file,
                                    "UUID": file,
                                    "Fecha Pago": fechaP,
                                    "Forma Pago": formaP,
                                    "Moneda Pago": monedaP,
                                    "Tipo_C Pago": tipo_cP,
                                    "Monto Pago": montoP,
                                    "# Operac Pago": nooperP,
                                    "Banco Pago": bcoordP,
                                    "Cta ordenante Pago": ctaordP,
                                    "Cta beneficiario Pago": ctabenP,
                                    "ID Doct": id_documento,
                                    "Serie": serie,
                                    "Folio": folio,
                                    "Moneda": moneda_dr,
                                    "Parcialidad": num_parcialidad,
                                    "ImpSaldoAnt": imp_saldo_ant,
                                    "ImpPagado": imp_pagado,
                                    "ImpSaldoInsoluto": imp_saldo_insoluto,
                                    "BaseDR": "N/A",
                                    "ImporteDR": "N/A",
                                    "ImpuestoDR": "N/A"
                                })

                except etree.XMLSyntaxError:
                    print(f"Invalid XML in file: {file_path}")
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")
                
    df = pd.DataFrame(file_list)
    return df


# Uso
year="2026"
empresa="Etal"
sentido="Emitidos"

directory_path = rf'C:\Users\RGARCIA\Desktop\CFDI_Python\{sentido}\{empresa}\{year}\Trabajo Pago'
df_files = list_files_in_directory(directory_path)

In [2]:


# Especificar el directorio de los archivos XML
d_path = rf'C:\Users\RGARCIA\Desktop\CFDI_Python\{sentido}\{empresa}\{year}'
output_csv_path = os.path.join(d_path, f'DC_Pago {sentido} {year}.csv')

df_files.to_csv(output_csv_path,index = False)